# SQL Queries
This notebook runs SQL queries against the SQLite database and saves each result as a CSV file.

**How it works:** We write a SQL query as a string, pass it to `pd.read_sql_query()` which runs it on the database and gives us back a DataFrame, then we save that to CSV.

In [13]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH    = Path('../sql/olist.db')
OUTPUT_DIR = Path('../docs/sql_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # create folder if it doesn't exist

# open the database connection
conn = sqlite3.connect(DB_PATH)

# helper function: run a query, print the result, and save to CSV
def run_query(sql, filename):
    df = pd.read_sql_query(sql, conn)
    df.to_csv(OUTPUT_DIR / filename, index=False)
    return df

print('Connected to database.')

Connected to database.


## Query 1 — Total Revenue
What is the total revenue across all orders?

In [14]:
run_query("""
SELECT ROUND(SUM(revenue), 2) AS total_revenue
FROM sales
""", '01_total_revenue.csv')

,total_revenue
0,15419773.75


## Query 2 — Monthly Revenue
How does revenue change month by month?

In [15]:
run_query("""
SELECT
    year,
    month,
    ROUND(SUM(revenue), 2) AS revenue
FROM sales
GROUP BY year, month
ORDER BY year, month
""", '02_monthly_revenue.csv')

,year,month,revenue
0,2016,9,143.46
1,2016,10,46490.66
2,2016,12,19.62
3,2017,1,127482.37
4,2017,2,271239.32
5,2017,3,414330.95
6,2017,4,390812.40
7,2017,5,566851.40
8,2017,6,490050.37
9,2017,7,566299.08


## Query 3 — Top 10 Categories by Revenue
Which product categories sell the most?

In [16]:
run_query("""
SELECT
    category_english,
    ROUND(SUM(revenue), 2)      AS revenue,
    COUNT(DISTINCT order_id)    AS orders
FROM sales
GROUP BY category_english
ORDER BY revenue DESC
LIMIT 10
""", '03_top_categories.csv')

,category_english,revenue,orders
0,health_beauty,1412089.53,8647
1,watches_gifts,1264333.12,5495
2,bed_bath_table,1225209.26,9272
3,sports_leisure,1118256.91,7530
4,computers_accessories,1032723.77,6530
5,furniture_decor,880329.92,6307
6,housewares,758392.25,5743
7,cool_stuff,691680.89,3559
8,auto,669454.75,3810
9,garden_tools,567145.68,3448


## Query 4 — Top 10 States by Revenue

In [17]:
run_query("""
SELECT
    customer_state,
    ROUND(SUM(revenue), 2)      AS revenue,
    COUNT(DISTINCT order_id)    AS orders
FROM sales
GROUP BY customer_state
ORDER BY revenue DESC
LIMIT 10
""", '04_top_states.csv')

,customer_state,revenue,orders
0,SP,5769703.15,40501
1,RJ,2055401.57,12350
2,MG,1818891.67,11354
3,RS,861472.79,5345
4,PR,781708.80,4923
5,SC,595127.78,3546
6,BA,591137.81,3256
7,DF,346123.35,2080
8,GO,334212.35,1957
9,ES,317657.93,1995


## Query 5 — Top 20 Customers

In [18]:
run_query("""
SELECT
    customer_unique_id,
    ROUND(SUM(revenue), 2)      AS total_spent,
    COUNT(DISTINCT order_id)    AS orders
FROM sales
GROUP BY customer_unique_id
ORDER BY total_spent DESC
LIMIT 20
""", '05_top_customers.csv')

,customer_unique_id,total_spent,orders
0,0a0a92112bd4c708ca5fde585afaa872,13664.08,1
1,da122df9eeddfedc1dc1f5349a1a690c,7571.63,2
2,763c8b1c9c68a0229c42c9fc6f662b93,7274.88,1
3,dc4802a71eae9be1dd28f5d788ceb526,6929.31,1
4,459bef486812aa25204be022145caa62,6922.21,1
5,ff4159b92c40ebe40454e3e6a7c35ed6,6726.66,1
6,4007669dec559734d6f53e029e360987,6081.54,1
7,eebb5dda148d3893cdaf5b5ca3040ccb,4764.34,1
8,48e1ac109decbb87765a3eade6854098,4681.78,1
9,c8460e4251689ba205045f3ea17884a1,4655.88,4


## Query 6 — Average Order Value (AOV)
On average, how much does a customer spend per order?

In [19]:
run_query("""
SELECT ROUND(SUM(revenue) / COUNT(DISTINCT order_id), 2) AS avg_order_value
FROM sales
""", '06_aov.csv')

,avg_order_value
0,159.83


## Query 7 — Year-over-Year Growth
How much did revenue grow from one year to the next?

`LAG()` is a window function that looks at the previous row — here it gives us last year's revenue so we can calculate the % change.

In [20]:
run_query("""
SELECT
    year,
    ROUND(SUM(revenue), 2)  AS revenue,
    -- LAG gets the previous year's total
    LAG(SUM(revenue)) OVER (ORDER BY year)  AS prev_year_revenue,
    -- calculate % growth vs previous year
    ROUND(
        100.0 * (SUM(revenue) - LAG(SUM(revenue)) OVER (ORDER BY year))
        / LAG(SUM(revenue)) OVER (ORDER BY year),
    2) AS growth_pct
FROM sales
GROUP BY year
ORDER BY year
""", '07_yoy_growth.csv')

,year,revenue,prev_year_revenue,growth_pct
0,2016,46653.74,NaN,NaN
1,2017,6921535.24,46653.74,14735.97
2,2018,8451584.77,6921535.24,22.11


## Query 8 — Top 10 Products by Revenue

In [21]:
run_query("""
SELECT
    product_id,
    ROUND(SUM(revenue), 2)  AS revenue,
    COUNT(*)                AS units_sold
FROM sales
GROUP BY product_id
ORDER BY revenue DESC
LIMIT 10
""", '08_top_products.csv')

,product_id,revenue,units_sold
0,bb50f2e236e5eea0100680137654686c,67258.03,194
1,d1c427060a0f73f6b889a5c7c61f2ac4,58957.31,332
2,6cdd53843498f92890544667809f1595,57933.73,153
3,99a4788cb24856965c36a24e339b6058,49907.50,477
4,3dd2a17168ec895c781a9191c1e95ad7,47876.06,272
5,d6160fb7873f184099d9bc95e30376af,47314.18,33
6,aca2eb7d00ea1a7b8ebd4e68314663af,44197.75,520
7,5f504b3a1c75b73d6151be81eb05bdc9,41725.81,63
8,25c38557cf793876c5abdd5931f922db,40311.95,38
9,53b36df67ebb7c41585e8d54d6772e08,39713.49,321


## Query 9 — Repeat vs One-Time Customers
How many customers bought more than once?

We use a **subquery**: first count orders per customer (inner query), then summarise those counts (outer query).

In [22]:
run_query("""
SELECT
    -- count customers who only ordered once
    SUM(CASE WHEN order_count = 1 THEN 1 ELSE 0 END)  AS one_time_customers,
    -- count customers who ordered more than once
    SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END)  AS repeat_customers,
    COUNT(*)                                            AS total_customers,
    -- what % of customers are repeat buyers?
    ROUND(100.0 * SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS repeat_rate_pct
FROM (
    -- subquery: how many orders did each customer place?
    SELECT customer_unique_id, COUNT(DISTINCT order_id) AS order_count
    FROM sales
    GROUP BY customer_unique_id
)
""", '09_repeat_customers.csv')

,one_time_customers,repeat_customers,total_customers,repeat_rate_pct
0,90557,2801,93358,3.0


## Query 10 — Quarterly Revenue

In [23]:
run_query("""
SELECT
    year,
    quarter,
    ROUND(SUM(revenue), 2)      AS revenue,
    COUNT(DISTINCT order_id)    AS orders
FROM sales
GROUP BY year, quarter
ORDER BY year, quarter
""", '10_quarterly_revenue.csv')

,year,quarter,revenue,orders
0,2016,3,143.46,1
1,2016,4,46510.28,266
2,2017,1,813052.64,4949
3,2017,2,1447714.17,8984
4,2017,3,1913208.93,12215
5,2017,4,2747559.50,17280
6,2018,1,3164654.11,20627
7,2018,2,3273631.74,19646
8,2018,3,2013298.92,12510


## Query 11 — ABC Product Classification
The Pareto / 80-20 rule: a small number of products usually drive most of the revenue.

- **Class A** — top products that account for the first 80% of revenue
- **Class B** — next products that bring it from 80% to 95%
- **Class C** — the rest

We use a **CTE (WITH clause)** to break the logic into steps that are easier to read.

In [24]:
run_query("""
-- Step 1: total revenue per product
WITH product_rev AS (
    SELECT product_id, SUM(revenue) AS revenue
    FROM sales
    GROUP BY product_id
),
-- Step 2: add a running total column (cumulative sum sorted by revenue descending)
ranked AS (
    SELECT
        product_id,
        revenue,
        SUM(revenue) OVER (ORDER BY revenue DESC) AS running_total,
        SUM(revenue) OVER ()                       AS grand_total
    FROM product_rev
)
-- Step 3: classify each product and count how many fall in each class
SELECT
    SUM(CASE WHEN running_total <= 0.80 * grand_total THEN 1 ELSE 0 END) AS class_A,
    SUM(CASE WHEN running_total >  0.80 * grand_total
             AND running_total <= 0.95 * grand_total THEN 1 ELSE 0 END) AS class_B,
    SUM(CASE WHEN running_total >  0.95 * grand_total THEN 1 ELSE 0 END) AS class_C,
    COUNT(*) AS total_products
FROM ranked
""", '11_abc_classification.csv')

conn.close()
print('All queries done. Results saved to', OUTPUT_DIR)

All queries done. Results saved to ..\docs\sql_results
